In [ ]:
import os, sys
print("cwd:", os.getcwd())
print("has workflow:", os.path.exists("03_core_algorithm/modules/benchmark_experiment_workflow.py"))
print("has export utils:", os.path.exists("03_core_algorithm/modules/benchmark_export_plot_utils.py"))
print("has base data:", os.path.exists("02_processed_data/classic/base"))
print("py:", sys.version)

cwd: /content
has workflow: False
has export utils: False
has base data: False
py: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
# Stable package install for notebook runs.
# Python 3.13 needs a newer OR-Tools wheel.
# No forced kernel kill here; restart kernel manually once after installation.
%pip install -q "ortools==9.15.6755" pandas numpy matplotlib vrplib
print("Packages installed. Please use: Kernel -> Restart Kernel, then Run All Cells.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 6.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
Packages installed. Please use: Kernel -> Restart Kernel, then Run All Cells.


In [ ]:
!pip install -q vrplib

In [ ]:
import math
import time
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import vrplib
from ortools.constraint_solver import pywrapcp, routing_enums_pb2
import json
from pathlib import Path

In [ ]:
def euclidean_distance(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])

def build_distance_matrix(coords):
    n = len(coords)
    dist = np.zeros((n, n), dtype=int)
    for i in range(n):
        for j in range(n):
            dist[i, j] = int(round(euclidean_distance(coords[i], coords[j])))
    return dist

def route_distance(route, dist_matrix):
    return sum(dist_matrix[route[i], route[i+1]] for i in range(len(route)-1))

def total_distance(routes, dist_matrix):
    return sum(route_distance(r, dist_matrix) for r in routes)

def check_feasibility(routes, demands, capacity, depot=0):
    for route in routes:
        load = sum(demands[node] for node in route if node != depot)
        if load > capacity:
            return False
    return True

In [ ]:
def evaluate_solver(instance, solver_fn, solver_name):
    dist_matrix = build_distance_matrix(instance["coords"])

    start = time.time()
    routes = solver_fn(instance)
    runtime = time.time() - start

    feasible = check_feasibility(
        routes,
        instance["demands"],
        instance["capacity"],
        instance["depot"]
    )

    cost = total_distance(routes, dist_matrix) if routes else None

    return {
        "instance": instance["name"],
        "solver": solver_name,
        "cost": cost,
        "runtime_sec": runtime,
        "feasible": feasible,
        "num_routes": len(routes),
        "routes": routes
    }

# **greedy_cvrp_solver**

In [ ]:
def greedy_cvrp_solver(instance):
    demands = instance["demands"]
    capacity = instance["capacity"]
    depot = instance["depot"]

    customers = list(range(1, len(demands)))
    routes = []
    current_route = [depot]
    remaining = capacity

    for c in customers:
        d = demands[c]
        if d <= remaining:
            current_route.append(c)
            remaining -= d
        else:
            current_route.append(depot)
            routes.append(current_route)
            current_route = [depot, c]
            remaining = capacity - d

    current_route.append(depot)
    routes.append(current_route)
    return routes

# **OR-Tools CVRP solver**

In [ ]:
def ortools_cvrp_solver(instance, time_limit_sec=3):
    coords = instance["coords"]
    demands = instance["demands"]
    capacity = instance["capacity"]
    depot = instance["depot"]

    dist_matrix = build_distance_matrix(coords)
    n = len(coords)

    num_vehicles = len(coords)

    manager = pywrapcp.RoutingIndexManager(n, num_vehicles, depot)
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        from_node = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)
        return int(dist_matrix[from_node][to_node])

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    def demand_callback(from_index):
        from_node = manager.IndexToNode(from_index)
        return int(demands[from_node])

    demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
    routing.AddDimensionWithVehicleCapacity(
        demand_callback_index,
        0,                       # no slack
        [capacity] * num_vehicles,
        True,                    # start cumul to zero
        "Capacity"
    )

    search_parameters = pywrapcp.DefaultRoutingSearchParameters()
    search_parameters.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
    search_parameters.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    search_parameters.time_limit.seconds = time_limit_sec

    solution = routing.SolveWithParameters(search_parameters)

    if solution is None:
        return []

    routes = []
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route = []
        while not routing.IsEnd(index):
            route.append(manager.IndexToNode(index))
            index = solution.Value(routing.NextVar(index))
        route.append(manager.IndexToNode(index))

        if len(route) > 2:
            routes.append(route)

    return routes

# **Nearest Neighbor CVRP**

In [ ]:
def nearest_neighbor_cvrp_solver(instance):
    coords = instance["coords"]
    demands = instance["demands"]
    capacity = instance["capacity"]
    depot = instance["depot"]

    dist_matrix = build_distance_matrix(coords)

    unvisited = set(range(1, len(coords)))
    routes = []

    while unvisited:
        route = [depot]
        current = depot
        remaining = capacity

        while True:
            feasible_candidates = [
                c for c in unvisited if demands[c] <= remaining
            ]

            if not feasible_candidates:
                break

            next_customer = min(
                feasible_candidates,
                key=lambda c: dist_matrix[current][c]
            )

            route.append(next_customer)
            unvisited.remove(next_customer)
            remaining -= demands[next_customer]
            current = next_customer

        route.append(depot)
        routes.append(route)

    return routes

# **heuristic_cvrp_solver**

In [ ]:
def nn_score(current, c, instance, remaining, dist_matrix):
    return dist_matrix[current][c]


def heuristic_cvrp_solver(instance, score_fn):
    coords = instance["coords"]
    demands = instance["demands"]
    capacity = instance["capacity"]
    depot = instance["depot"]

    dist_matrix = build_distance_matrix(coords)
    unvisited = set(range(1, len(coords)))
    routes = []

    while unvisited:
        route = [depot]
        current = depot
        remaining = capacity

        while True:
            feasible_candidates = [
                c for c in unvisited if demands[c] <= remaining
            ]

            if not feasible_candidates:
                break

            next_customer = min(
                feasible_candidates,
                key=lambda c: score_fn(current, c, instance, remaining, dist_matrix)
            )

            route.append(next_customer)
            unvisited.remove(next_customer)
            remaining -= demands[next_customer]
            current = next_customer

        route.append(depot)
        routes.append(route)

    return routes


def nearest_neighbor_v2(instance):
    return heuristic_cvrp_solver(instance, nn_score)

In [ ]:
def make_score_fn_from_expression(expr):
    def score_fn(current, c, instance, remaining, dist_matrix):
        return eval(
            expr,
            {"__builtins__": {}},
            {
                "current": current,
                "c": c,
                "instance": instance,
                "remaining": remaining,
                "dist_matrix": dist_matrix,
            }
        )
    return score_fn

# **Upload the dataset** (zip file)

In [ ]:
from pathlib import Path

DATA_ROOT = Path("/content/CVRP/02_processed_data")
DATA_ROOT

Saving classic_cvrp_dataset.zip to classic_cvrp_dataset.zip


**Use repository data under `02_processed_data/` instead of delivery zip packages.**

In [ ]:
base_dir = Path("data/processed/base")
json_files = sorted(base_dir.glob("*.base.json"))

print(json_files[0])

with open(json_files[0], "r", encoding="utf-8") as f:
    sample = json.load(f)

print(type(sample))
print(sample.keys())

print(sample["depot"])
print(sample["customers"][0])

data/processed/base/A-n32-k5.base.json
<class 'dict'>
dict_keys(['instance_id', 'set_id', 'type', 'dimension', 'vehicle_capacity', 'vehicle_count_hint', 'depot', 'customers', 'node_id_order', 'distance_matrix', 'distance_metric', 'known_opt_cost', 'known_opt_routes'])
{'id': 1, 'x': 82.0, 'y': 76.0, 'demand': 0}
{'id': 2, 'x': 96.0, 'y': 44.0, 'demand': 19}


In [ ]:
def load_base_instance(json_path):
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    depot = data["depot"]
    customers = data["customers"]

    coords = [[depot["x"], depot["y"]]]
    demands = [0]

    for c in customers:
        coords.append([c["x"], c["y"]])
        demands.append(c["demand"])

    instance = {
        "name": data["instance_id"],
        "depot": 0,
        "coords": coords,
        "demands": demands,
        "capacity": data["vehicle_capacity"],
        "num_nodes": len(coords),
        "distance_matrix": data["distance_matrix"],
        "raw": data, # other keys
    }
    return instance

In [ ]:
def load_multiple_base_instances(base_dir, limit=None):
    json_files = sorted(Path(base_dir).glob("*.base.json"))
    if limit is not None:
        json_files = json_files[:limit]
    return [load_base_instance(p) for p in json_files]

**Benchmark loading config (set `BENCHMARK_LIMIT=None` for full run)**

In [ ]:
# P1: formal benchmark should avoid tiny debug subset.
# Use BENCHMARK_LIMIT=None for full set, or an integer for quick smoke tests.
BENCHMARK_LIMIT = None
instances = load_multiple_base_instances("data/processed/base", limit=BENCHMARK_LIMIT)
print(len(instances))
print(instances[0]["name"])

95
A-n32-k5


In [ ]:
def evaluate_expression_on_instances(instances, expr):
    score_fn = make_score_fn_from_expression(expr)
    rows = []

    for inst in instances:
        result = evaluate_solver(
            instance=inst,
            solver_name=expr,
            solver_fn=lambda instance: heuristic_cvrp_solver(instance, score_fn)
        )
        rows.append({
            "instance": inst["name"],
            "expression": expr,
            "cost": result["cost"],
            "runtime_sec": result["runtime_sec"],
            "feasible": result["feasible"],
            "num_routes": result["num_routes"],
        })

    return pd.DataFrame(rows)

In [ ]:
json_files = sorted(Path("data/processed/base").glob("*.base.json"))

bad_raw = []

for p in json_files:
    with open(p, "r", encoding="utf-8") as f:
        data = json.load(f)

    depot = data["depot"]
    if depot.get("x") is None or depot.get("y") is None:
        bad_raw.append((p.name, "depot", depot))

    for c in data["customers"]:
        if c.get("x") is None or c.get("y") is None:
            bad_raw.append((p.name, c.get("id"), c))
            break

print("raw bad count:", len(bad_raw))
print("first 10 raw bad:", bad_raw[:10])

raw bad count: 4
first 10 raw bad: [('E-n13-k4.base.json', 'depot', {'id': 1, 'x': None, 'y': None, 'demand': 0}), ('E-n13-k4.base.json', 2, {'id': 2, 'x': None, 'y': None, 'demand': 1200}), ('E-n31-k7.base.json', 'depot', {'id': 1, 'x': None, 'y': None, 'demand': 0}), ('E-n31-k7.base.json', 2, {'id': 2, 'x': None, 'y': None, 'demand': 24})]


In [ ]:
bad_names = sorted(set(x[0] for x in bad_raw))
print("bad files:", bad_names)

instances = [
    inst for inst in instances
    if inst["name"] + ".base.json" not in bad_names
]

print("clean instances:", len(instances))

df = evaluate_expression_on_instances(
    instances[:3],
    "dist_matrix[current][c]"
)
df.head()

bad files: ['E-n13-k4.base.json', 'E-n31-k7.base.json']
clean instances: 93


,instance,expression,cost,runtime_sec,feasible,num_routes
0,A-n32-k5,dist_matrix[current][c],1145,0.008811,True,5
1,A-n33-k5,dist_matrix[current][c],977,0.012410,True,5
2,A-n33-k6,dist_matrix[current][c],1042,0.012031,True,6


In [ ]:
def summarize_expression_results(df):
    out = (
        df.assign(feasible_num=df["feasible"].astype(int))
          .groupby("expression")
          .agg(
              num_instances=("instance", "count"),
              feasible_rate=("feasible_num", "mean"),
              avg_cost=("cost", "mean"),
              avg_runtime_sec=("runtime_sec", "mean"),
              avg_num_routes=("num_routes", "mean"),
          )
          .reset_index()
          .sort_values(["feasible_rate", "avg_cost"], ascending=[False, True])
    )
    return out

In [ ]:
def evaluate_named_solver_on_instances(instances, solver_name, solver_fn):
    rows = []
    for inst in instances:
        result = evaluate_solver(
            instance=inst,
            solver_name=solver_name,
            solver_fn=solver_fn
        )
        rows.append({
            "instance": inst["name"],
            "expression": solver_name,
            "cost": result["cost"],
            "runtime_sec": result["runtime_sec"],
            "feasible": result["feasible"],
            "num_routes": result["num_routes"],
        })
    return pd.DataFrame(rows)

In [ ]:
df_nn = evaluate_named_solver_on_instances(instances, "nearest_neighbor", nearest_neighbor_v2)
df_greedy = evaluate_named_solver_on_instances(instances, "greedy", greedy_cvrp_solver)
df_ortools = evaluate_named_solver_on_instances(instances, "ortools", ortools_cvrp_solver)

In [ ]:
all_df_full = pd.concat(
    [
        df_nn,
        df_greedy,
        df_ortools
    ],
    ignore_index=True
)

summary_full = summarize_expression_results(all_df_full)
summary_full

,expression,num_instances,feasible_rate,avg_cost,avg_runtime_sec,avg_num_routes
2,ortools,93,1.0,910.268817,3.006427,7.505376
1,nearest_neighbor,93,1.0,1201.010753,0.003336,7.483871
0,greedy,93,1.0,2321.376344,0.000010,8.000000


In [ ]:
def search_expressions_multi(instances, candidate_expressions, top_k=5):
    all_df = pd.concat(
        [evaluate_expression_on_instances(instances, expr) for expr in candidate_expressions],
        ignore_index=True
    )
    summary_df = summarize_expression_results(all_df)
    top_df = summary_df.head(top_k)
    return all_df, summary_df, top_df

In [ ]:
candidate_expressions = [
    "dist_matrix[current][c]",
    "dist_matrix[current][c] - 2 * instance['demands'][c]",
    "dist_matrix[current][c] + 0.3 * dist_matrix[c][instance['depot']]",
    "dist_matrix[current][c] - instance['demands'][c]",
    "dist_matrix[current][c] + instance['demands'][c]",
]

detail_df, summary_df, top_df = search_expressions_multi(instances, candidate_expressions, top_k=5)
top_df

,expression,num_instances,feasible_rate,avg_cost,avg_runtime_sec,avg_num_routes
0,dist_matrix[current][c],93,1.0,1201.010753,0.042844,7.483871
1,dist_matrix[current][c] + 0.3 * dist_matrix[c]...,93,1.0,1218.107527,0.065335,7.430108
4,dist_matrix[current][c] - instance['demands'][c],93,1.0,1262.225806,0.047869,7.408602
3,dist_matrix[current][c] - 2 * instance['demand...,93,1.0,1333.354839,0.044433,7.419355
2,dist_matrix[current][c] + instance['demands'][c],93,1.0,1347.225806,0.040706,7.666667


# ***duplicate checking***

In [ ]:
import ast
import math

# Advanced duplicate checking (from 03_core_algorithm/methods_advanced/duplicate_checking.py)
class Canonicalizer(ast.NodeTransformer):
    """Normalize arithmetic expression AST to reduce superficial syntax differences."""

    COMMUTATIVE_OPS = (ast.Add, ast.Mult)

    def visit_BinOp(self, node):
        self.generic_visit(node)

        if isinstance(node.op, ast.Add):
            if isinstance(node.right, ast.Constant) and node.right.value == 0:
                return node.left
            if isinstance(node.left, ast.Constant) and node.left.value == 0:
                return node.right

        if isinstance(node.op, ast.Mult):
            if isinstance(node.right, ast.Constant) and node.right.value == 1:
                return node.left
            if isinstance(node.left, ast.Constant) and node.left.value == 1:
                return node.right

        if isinstance(node.op, self.COMMUTATIVE_OPS):
            left_dump = ast.dump(node.left)
            right_dump = ast.dump(node.right)
            if right_dump < left_dump:
                node.left, node.right = node.right, node.left

        return node


def canonicalize_expr(expr: str) -> str:
    try:
        tree = ast.parse(expr, mode="eval")
        tree = Canonicalizer().visit(tree)
        ast.fix_missing_locations(tree)
        return ast.unparse(tree)
    except Exception:
        return str(expr).strip()


def make_behavior_fingerprint(expr, probe_instances, eval_fn, rounding=2):
    """Evaluate expr on a probe set and build a behavior fingerprint."""
    try:
        detail_df = eval_fn(expr, probe_instances).copy()

        if "instance" in detail_df.columns:
            detail_df = detail_df.sort_values("instance").reset_index(drop=True)
        elif "instance_id" in detail_df.columns:
            detail_df = detail_df.sort_values("instance_id").reset_index(drop=True)

        feasible_vec = tuple(detail_df["feasible"].astype(int).tolist())
        cost_vec = tuple(np.round(detail_df["cost"].astype(float).values, rounding)) if "cost" in detail_df.columns else ()
        routes_vec = tuple(np.round(detail_df["num_routes"].astype(float).values, rounding)) if "num_routes" in detail_df.columns else ()

        feasible_rate = round(float(detail_df["feasible"].mean()), 3)
        avg_cost = round(float(detail_df["cost"].mean()), rounding) if "cost" in detail_df.columns else math.inf
        avg_num_routes = round(float(detail_df["num_routes"].mean()), rounding) if "num_routes" in detail_df.columns else math.inf

        return (feasible_vec, cost_vec, routes_vec, feasible_rate, avg_cost, avg_num_routes)
    except Exception:
        return ("ERROR", expr)


def dedup_candidates_advanced(candidate_expressions, probe_instances, eval_fn, use_behavioral=True):
    kept = []
    seen_canonical = set()
    seen_behavior = set()

    for expr in candidate_expressions:
        canon = canonicalize_expr(expr)
        if canon in seen_canonical:
            continue

        if use_behavioral:
            fp = make_behavior_fingerprint(canon, probe_instances, eval_fn)
            if fp in seen_behavior:
                continue
            seen_behavior.add(fp)

        seen_canonical.add(canon)
        kept.append(canon)

    return kept


def dedup_expressions(
    candidate_expressions,
    probe_instances=None,
    eval_fn=None,
    use_behavioral=False,
):
    """
    Backward-compatible dedup API.
    - default: syntax-level canonicalized dedup
    - optional: add behavioral dedup on probe instances
    """
    if probe_instances is not None and eval_fn is not None:
        return dedup_candidates_advanced(
            candidate_expressions,
            probe_instances=probe_instances,
            eval_fn=eval_fn,
            use_behavioral=use_behavioral,
        )

    kept = []
    seen_canonical = set()
    for expr in candidate_expressions:
        canon = canonicalize_expr(expr)
        if canon in seen_canonical:
            continue
        seen_canonical.add(canon)
        kept.append(canon)
    return kept



# ***complexity***

In [ ]:
def expression_complexity(expr):
    """
    Advanced complexity estimate:
    - AST node count (structure complexity)
    - operation-weighted penalty
    - variable/reference diversity
    """
    expr = str(expr)
    try:
        tree = ast.parse(expr, mode="eval")
    except Exception:
        ops = ['+', '-', '*', '/', '%']
        op_count = sum(expr.count(op) for op in ops)
        return len(expr) + 5 * op_count

    node_count = 0
    op_penalty = 0
    names = set()

    op_weights = {
        ast.Add: 1,
        ast.Sub: 1,
        ast.Mult: 2,
        ast.Div: 3,
        ast.Mod: 3,
        ast.Pow: 4,
        ast.USub: 1,
    }

    for node in ast.walk(tree):
        node_count += 1
        if isinstance(node, ast.Name):
            names.add(node.id)
        elif isinstance(node, ast.Subscript):
            op_penalty += 2
        elif isinstance(node, ast.Call):
            op_penalty += 8

        for op_t, w in op_weights.items():
            if isinstance(node, op_t):
                op_penalty += w

    diversity_penalty = len(names)
    return int(node_count + op_penalty + diversity_penalty)



In [ ]:
def filter_expressions_by_complexity(expressions, max_complexity=None):
    if max_complexity is None:
        return list(expressions)

    kept = []
    for expr in expressions:
        c = expression_complexity(expr)
        if c <= max_complexity:
            kept.append(expr)
    return kept



# ***novelty***

In [ ]:
def make_behavior_signature_from_summary_row(
    row,
    cost_round=2,
    route_round=2,
    feas_round=3,
    complexity_bucket=5,
):
    avg_cost = row.get("avg_cost", np.inf)
    avg_num_routes = row.get("avg_num_routes", np.inf)
    feasible_rate = row.get("feasible_rate", 0.0)
    complexity = row.get("complexity", np.inf)

    if pd.isna(avg_cost):
        avg_cost = np.inf
    if pd.isna(avg_num_routes):
        avg_num_routes = np.inf
    if pd.isna(feasible_rate):
        feasible_rate = 0.0
    if pd.isna(complexity):
        complexity = np.inf

    complexity_bin = complexity if not np.isfinite(complexity) else int(complexity // complexity_bucket)

    return (
        round(float(avg_cost), cost_round) if np.isfinite(avg_cost) else np.inf,
        round(float(avg_num_routes), route_round) if np.isfinite(avg_num_routes) else np.inf,
        round(float(feasible_rate), feas_round),
        complexity_bin,
    )



In [ ]:
def add_novelty_columns(summary_df, archive_signatures=None):
    if archive_signatures is None:
        archive_signatures = set()

    df = summary_df.copy()
    df["behavior_signature"] = df.apply(make_behavior_signature_from_summary_row, axis=1)
    df["is_novel"] = ~df["behavior_signature"].isin(archive_signatures)
    return df



In [ ]:
def update_archive_signatures(summary_df, archive_signatures=None, only_novel=True):
    if archive_signatures is None:
        archive_signatures = set()

    df = summary_df.copy()
    if "behavior_signature" not in df.columns:
        df["behavior_signature"] = df.apply(make_behavior_signature_from_summary_row, axis=1)

    if only_novel and "is_novel" in df.columns:
        df = df[df["is_novel"]]

    for sig in df["behavior_signature"]:
        archive_signatures.add(sig)

    return archive_signatures



**multi-objective ranking**


In [ ]:
def _safe_minmax_norm(series, larger_is_better=False):
    s = pd.to_numeric(series, errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan)
    if s.notna().sum() == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)

    lo, hi = s.min(), s.max()
    if hi == lo:
        norm = pd.Series(np.zeros(len(series)), index=series.index)
    else:
        norm = (s - lo) / (hi - lo)
    norm = norm.fillna(1.0)

    if larger_is_better:
        return 1.0 - norm
    return norm


def sort_expression_summary(summary_df):
    """
    Multi-objective ranking:
    - Primary lexicographic order: feasible_rate (desc), avg_cost/avg_num_routes/complexity (asc)
    - Tie-breaker: weighted normalized score
    """
    df = summary_df.copy()

    if "complexity" not in df.columns:
        df["complexity"] = df["expression"].apply(expression_complexity)

    n_feasible = _safe_minmax_norm(df["feasible_rate"], larger_is_better=True)
    n_cost = _safe_minmax_norm(df["avg_cost"], larger_is_better=False)
    n_routes = _safe_minmax_norm(df["avg_num_routes"], larger_is_better=False)
    n_complexity = _safe_minmax_norm(df["complexity"], larger_is_better=False)

    df["mo_score"] = (
        0.55 * n_feasible +
        0.25 * n_cost +
        0.15 * n_routes +
        0.05 * n_complexity
    )

    return df.sort_values(
        by=["feasible_rate", "avg_cost", "avg_num_routes", "complexity", "mo_score"],
        ascending=[False, True, True, True, True]
    ).reset_index(drop=True)



**mock generation**

In [ ]:
def generate_mock_expression_variants(seed_expr, n=8):
    variants = set()

    demand_terms = [
        "instance['demands'][c]",
        f"2 * instance['demands'][c]",
        f"0.5 * instance['demands'][c]",
    ]

    depot_terms = [
        "dist_matrix[c][instance['depot']]",
        f"0.3 * dist_matrix[c][instance['depot']]",
        f"0.5 * dist_matrix[c][instance['depot']]",
    ]

    base_terms = [
        "dist_matrix[current][c]",
    ]

    templates = [
        "{base}",
        "{base} - {demand}",
        "{base} + {demand}",
        "{base} - {depot}",
        "{base} + {depot}",
        "{base} - {demand} + {depot}",
        "{base} - {demand} - {depot}",
        "{base} + {demand} + {depot}",
    ]

    for _ in range(n * 3):
        tpl = random.choice(templates)
        expr = tpl.format(
            base=random.choice(base_terms),
            demand=random.choice(demand_terms),
            depot=random.choice(depot_terms),
        )
        variants.add(expr)
        if len(variants) >= n:
            break

    variants.add(seed_expr)
    return list(variants)

In [ ]:
def generate_mock_candidates_from_top_expressions(top_expressions, variants_per_expr=8):
    all_candidates = []
    for expr in top_expressions:
        candidates = generate_mock_expression_variants(expr, n=variants_per_expr)
        all_candidates.extend(candidates)
    return all_candidates

# **LLM**

In [ ]:
!pip -q install --upgrade openai

In [ ]:
import json
import re

In [ ]:
import openai

API_KEY = "Please use your api"
DEFAULT_LLM_MODEL = "gpt-5-nano"
HOST_URL = "https://api.bltcy.ai"
client = openai.OpenAI(
    base_url=f"{HOST_URL}/v1",
    api_key=API_KEY,
    timeout=120,
)

**prompt**

In [ ]:
def build_expression_generation_prompt(
    top_expressions,
    n_per_expr=5,
    max_total=20,
):
    seed_block = "\n".join([f"- {expr}" for expr in top_expressions])

    return f"""
Generate new candidate Python score expressions for a CVRP heuristic.

Lower score is better.

Available variables only:
- dist_matrix[current][c]
- dist_matrix[c][instance['depot']]
- instance['demands'][c]
- remaining

Seed expressions:
{seed_block}

Rules:
1. Return ONLY JSON.
2. JSON format must be:
{{"expressions": ["expr1", "expr2", "..."]}}
3. At most {max_total} expressions.
4. Expressions must be single-line valid Python arithmetic expressions.
5. No explanations.
6. No markdown.
7. No extra keys.
8. Prefer small local variations of the seed expressions.

Good examples:
- "dist_matrix[current][c] - instance['demands'][c]"
- "dist_matrix[current][c] + 0.2 * dist_matrix[c][instance['depot']]"
- "dist_matrix[current][c] - 0.5 * instance['demands'][c]"
""".strip()

In [ ]:
def try_parse_json_object(text):
    text = text.strip()

    try:
        return json.loads(text)
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        try:
            return json.loads(candidate)
        except Exception:
            pass

    return None


def extract_expressions_from_response_text(text):
    obj = try_parse_json_object(text)
    if obj is None:
        return []

    expressions = obj.get("expressions", [])
    if not isinstance(expressions, list):
        return []

    cleaned = []
    for expr in expressions:
        if isinstance(expr, str):
            cleaned.append(expr)
    return cleaned

In [ ]:
ALLOWED_PATTERN = re.compile(
    r"^[0-9a-zA-Z_\[\]\(\)\{\}\.\+\-\*/,'\" :><=!?&|]+$"
)

FORBIDDEN_SUBSTRINGS = [
    "__",
    "import",
    "exec",
    "eval",
    "open(",
    "os.",
    "sys.",
    "subprocess",
    "lambda",
    "for ",
    "while ",
    "class ",
    "def ",
    ";",
    "\n",
]

def normalize_expression(expr):
    if expr is None:
        return None
    expr = str(expr).strip()
    expr = expr.replace("`", "")
    expr = expr.replace("\u200b", "")
    return expr


def is_safe_expression(expr):
    expr = normalize_expression(expr)
    if not expr:
        return False

    if len(expr) > 200:
        return False

    if not ALLOWED_PATTERN.match(expr):
        return False

    lower_expr = expr.lower()
    for bad in FORBIDDEN_SUBSTRINGS:
        if bad.lower() in lower_expr:
            return False

    return True


def filter_valid_expressions(expressions, max_complexity=None):
    kept = []
    for expr in expressions:
        expr = normalize_expression(expr)
        if not is_safe_expression(expr):
            continue
        if max_complexity is not None and expression_complexity(expr) > max_complexity:
            continue
        kept.append(expr)
    return dedup_expressions(kept)

In [ ]:
def call_openai_for_expressions_chat(
    client,
    top_expressions,
    n_per_expr=5,
    model_name=DEFAULT_LLM_MODEL,
    temperature=0.4,
    max_tokens=3000,
):
    prompt = build_expression_generation_prompt(
        top_expressions=top_expressions,
        n_per_expr=n_per_expr,
        max_total=max(8, len(top_expressions) * n_per_expr),
    )

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {
                "role": "system",
                "content": (
                    "You generate candidate Python score expressions for CVRP search. "
                    "Return only valid JSON."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        temperature=temperature,
        max_completion_tokens=max_tokens,
    )

    msg = response.choices[0].message
    print("finish_reason:", response.choices[0].finish_reason)
    print("message:", msg)
    print("content repr:", repr(msg.content))

    text = msg.content.strip() if msg.content else ""
    return text, response

In [ ]:
def generate_candidates_with_llm(
    client,
    top_expressions,
    n_per_expr=5,
    model_name=DEFAULT_LLM_MODEL,
    temperature=0.4,
    max_complexity=None,
    verbose=True,
):
    raw_text, response = call_openai_for_expressions_chat(
        client=client,
        top_expressions=top_expressions,
        n_per_expr=n_per_expr,
        model_name=model_name,
        temperature=temperature,
    )

    raw_candidates = extract_expressions_from_response_text(raw_text)
    valid_candidates = filter_valid_expressions(
        raw_candidates,
        max_complexity=max_complexity
    )

    merged = dedup_expressions(list(top_expressions) + list(valid_candidates))

    if verbose:
        print("model:", model_name)
        print("finish_reason:", response.choices[0].finish_reason)
        print("LLM raw text:")
        print(raw_text[:1000])
        print("-" * 80)
        print("Parsed expressions:", len(raw_candidates))
        print("Valid expressions :", len(valid_candidates))
        print("Merged pool size  :", len(merged))

    return merged

In [ ]:
def evaluate_expression_list_on_instances(instances, expressions):
    expressions = list(expressions)
    if len(expressions) == 0:
        return pd.DataFrame()

    dfs = []
    for expr in expressions:
        df_expr = evaluate_expression_on_instances(instances, expr)
        dfs.append(df_expr)

    return pd.concat(dfs, ignore_index=True)

In [ ]:
def run_one_search_round(
    instances,
    candidate_expressions,
    archive_signatures=None,
    max_complexity=None,
    require_novel=False,
    top_k=5,
):
    if archive_signatures is None:
        archive_signatures = set()

    probe_instances = list(instances[:min(5, len(instances))])
    candidate_expressions = dedup_expressions(
        candidate_expressions,
        probe_instances=probe_instances,
        eval_fn=evaluate_expression_list_on_instances,
        use_behavioral=True,
    )

    candidate_expressions = filter_expressions_by_complexity(
        candidate_expressions,
        max_complexity=max_complexity
    )

    if len(candidate_expressions) == 0:
        return {
            "detail_df": pd.DataFrame(),
            "summary_df": pd.DataFrame(),
            "top_df": pd.DataFrame(),
            "archive_signatures": archive_signatures,
        }

    detail_df = evaluate_expression_list_on_instances(instances, candidate_expressions)

    if len(detail_df) == 0:
        return {
            "detail_df": pd.DataFrame(),
            "summary_df": pd.DataFrame(),
            "top_df": pd.DataFrame(),
            "archive_signatures": archive_signatures,
        }

    summary_df = summarize_expression_results(detail_df)

    if "complexity" not in summary_df.columns:
        summary_df["complexity"] = summary_df["expression"].apply(expression_complexity)

    summary_df = add_novelty_columns(summary_df, archive_signatures=archive_signatures)

    if require_novel:
        summary_df = summary_df[summary_df["is_novel"]].copy()

    if len(summary_df) == 0:
        return {
            "detail_df": detail_df,
            "summary_df": pd.DataFrame(),
            "top_df": pd.DataFrame(),
            "archive_signatures": archive_signatures,
        }

    summary_df = sort_expression_summary(summary_df)
    top_df = summary_df.head(top_k).reset_index(drop=True)

    new_archive = update_archive_signatures(
        top_df,
        archive_signatures=archive_signatures,
        only_novel=False
    )

    return {
        "detail_df": detail_df,
        "summary_df": summary_df,
        "top_df": top_df,
        "archive_signatures": new_archive,
    }


In [ ]:
def search_expressions_outer_loop(
    instances,
    seed_expressions,
    client=None,
    num_rounds=3,
    variants_per_expr=8,
    max_complexity=None,
    require_novel=False,
    top_k_per_round=5,
    generation_mode="llm",   # "mock" or "llm"
    llm_model_name=DEFAULT_LLM_MODEL,
    llm_temperature=0.4,
    verbose=True,
):
    archive_signatures = set()
    round_summaries = []
    round_tops = []
    all_detail_dfs = []

    current_pool = list(seed_expressions)

    for round_idx in range(num_rounds):
        result = run_one_search_round(
            instances=instances,
            candidate_expressions=current_pool,
            archive_signatures=archive_signatures,
            max_complexity=max_complexity,
            require_novel=require_novel,
            top_k=top_k_per_round,
        )

        detail_df = result["detail_df"]
        summary_df = result["summary_df"]
        top_df = result["top_df"]
        archive_signatures = result["archive_signatures"]

        if len(detail_df) > 0:
            detail_df = detail_df.copy()
            detail_df["round_idx"] = round_idx
            all_detail_dfs.append(detail_df)

        if len(summary_df) > 0:
            summary_df = summary_df.copy()
            summary_df["round_idx"] = round_idx
            round_summaries.append(summary_df)

        if len(top_df) > 0:
            top_df = top_df.copy()
            top_df["round_idx"] = round_idx
            round_tops.append(top_df)

        if verbose:
            print(f"Round {round_idx}:")
            print(f"  input pool size = {len(current_pool)}")
            print(f"  evaluated = {len(summary_df)}")
            print(f"  top kept = {len(top_df)}")
            if len(top_df) > 0:
                print(
                    top_df[
                        ["expression", "feasible_rate", "avg_cost", "avg_num_routes", "complexity", "is_novel"]
                    ].head(top_k_per_round)
                )
            print("-" * 80)

        if len(top_df) == 0:
            current_pool = []
            break

        next_seed_expressions = top_df["expression"].tolist()

        if generation_mode == "mock":
            current_pool = generate_mock_candidates_from_top_expressions(
                next_seed_expressions,
                variants_per_expr=variants_per_expr
            )
        elif generation_mode == "llm":
            if client is None:
                raise ValueError("generation_mode='llm' must input client")
            current_pool = generate_candidates_with_llm(
                client=client,
                top_expressions=next_seed_expressions,
                n_per_expr=variants_per_expr,
                model_name=llm_model_name,
                temperature=llm_temperature,
                max_complexity=max_complexity,
                verbose=verbose,
            )
        else:
            raise ValueError(f"Unknown generation_mode: {generation_mode}")

    all_round_summary_df = pd.concat(round_summaries, ignore_index=True) if round_summaries else pd.DataFrame()
    all_round_top_df = pd.concat(round_tops, ignore_index=True) if round_tops else pd.DataFrame()
    all_detail_df = pd.concat(all_detail_dfs, ignore_index=True) if all_detail_dfs else pd.DataFrame()

    return {
        "all_detail_df": all_detail_df,
        "all_round_summary_df": all_round_summary_df,
        "all_round_top_df": all_round_top_df,
        "archive_signatures": archive_signatures,
    }

In [ ]:
seed_expressions = [
    "dist_matrix[current][c]",
    "dist_matrix[current][c] - 2 * instance['demands'][c]",
    "dist_matrix[current][c] + 0.3 * dist_matrix[c][instance['depot']]",
    "dist_matrix[current][c] - instance['demands'][c]",
    "dist_matrix[current][c] + instance['demands'][c]",
]

In [ ]:
outer_result = search_expressions_outer_loop(
    instances=instances[:3],              # small test
    seed_expressions=seed_expressions,
    client=client,
    num_rounds=1,
    variants_per_expr=2,
    max_complexity=None,
    require_novel=False,
    top_k_per_round=2,
    generation_mode="llm",
    llm_model_name=DEFAULT_LLM_MODEL,
    llm_temperature=0.4,
    verbose=True,
)

Round 0:
  input pool size = 5
  evaluated = 5
  top kept = 2
                                          expression  feasible_rate  avg_cost  \
0   dist_matrix[current][c] - instance['demands'][c]            1.0     993.0   
1  0.3 * dist_matrix[c][instance['depot']] + dist...            1.0    1051.0   

   avg_num_routes  complexity  is_novel  
0        5.333333          35      True  
1        5.333333          46      True  
--------------------------------------------------------------------------------
finish_reason: stop
message: ChatCompletionMessage(content='{"expressions": ["dist_matrix[current][c] - instance[\'demands\'][c]", "dist_matrix[current][c] + 0.3 * dist_matrix[c][instance[\'depot\']]", "dist_matrix[current][c] - 0.3 * instance[\'demands\'][c]", "0.5 * dist_matrix[c][instance[\'depot\']] + dist_matrix[current][c]", "dist_matrix[current][c] - 0.5 * dist_matrix[c][instance[\'depot\']]", "dist_matrix[current][c] - 0.2 * instance[\'demands\'][c] + remaining", "0.4 * dist

In [ ]:
all_detail_outer = outer_result["all_detail_df"]
all_summary_outer = outer_result["all_round_summary_df"]
all_top_outer = outer_result["all_round_top_df"]

In [ ]:
all_top_outer

,expression,num_instances,feasible_rate,avg_cost,avg_runtime_sec,avg_num_routes,complexity,behavior_signature,is_novel,mo_score,round_idx
0,dist_matrix[current][c] - instance['demands'][c],10,1.0,1073.8,0.021621,5.3,35,"(1073.8, 5.3, 1.0, 7)",True,0.580357,0
1,dist_matrix[current][c],10,1.0,1080.4,0.010653,5.3,18,"(1080.4, 5.3, 1.0, 3)",True,0.573775,0
2,0.3 * dist_matrix[c][instance['depot']] + dist...,10,1.0,1112.0,0.030206,5.3,46,"(1112.0, 5.3, 1.0, 9)",True,0.737608,0
3,dist_matrix[current][c] - 2 * instance['demand...,10,1.0,1129.2,0.012282,5.3,40,"(1129.2, 5.3, 1.0, 8)",True,0.788853,0
4,dist_matrix[current][c] - dist_matrix[c][insta...,10,1.0,1006.2,0.012390,5.3,41,"(1006.2, 5.3, 1.0, 8)",True,0.573469,1
5,dist_matrix[current][c] - 0.5 * dist_matrix[c]...,10,1.0,1063.1,0.016217,5.3,46,"(1063.1, 5.3, 1.0, 9)",True,0.658577,1
6,dist_matrix[current][c] - instance['demands'][c],10,1.0,1073.8,0.038892,5.3,35,"(1073.8, 5.3, 1.0, 7)",False,0.662398,1
7,dist_matrix[current][c] - 0.3 * dist_matrix[c]...,10,1.0,1079.8,0.053881,5.3,46,"(1079.8, 5.3, 1.0, 9)",True,0.682058,1


In [ ]:
all_summary_outer.sort_values(
    by=["feasible_rate", "avg_cost", "avg_num_routes", "complexity"],
    ascending=[False, True, True, True]
).head(20)

,expression,num_instances,feasible_rate,avg_cost,avg_runtime_sec,avg_num_routes,complexity,behavior_signature,is_novel,mo_score,round_idx
5,dist_matrix[current][c] - dist_matrix[c][insta...,10,1.0,1006.2,0.012390,5.3,41,"(1006.2, 5.3, 1.0, 8)",True,0.573469,1
6,dist_matrix[current][c] - 0.5 * dist_matrix[c]...,10,1.0,1063.1,0.016217,5.3,46,"(1063.1, 5.3, 1.0, 9)",True,0.658577,1
0,dist_matrix[current][c] - instance['demands'][c],10,1.0,1073.8,0.021621,5.3,35,"(1073.8, 5.3, 1.0, 7)",True,0.580357,0
7,dist_matrix[current][c] - instance['demands'][c],10,1.0,1073.8,0.038892,5.3,35,"(1073.8, 5.3, 1.0, 7)",False,0.662398,1
8,dist_matrix[current][c] - 0.3 * dist_matrix[c]...,10,1.0,1079.8,0.053881,5.3,46,"(1079.8, 5.3, 1.0, 9)",True,0.682058,1
1,dist_matrix[current][c],10,1.0,1080.4,0.010653,5.3,18,"(1080.4, 5.3, 1.0, 3)",True,0.573775,0
9,dist_matrix[current][c],10,1.0,1080.4,0.021505,5.3,18,"(1080.4, 5.3, 1.0, 3)",False,0.654331,1
10,0.3 * dist_matrix[c][instance['depot']] + (dis...,10,1.0,1088.3,0.021301,5.3,67,"(1088.3, 5.3, 1.0, 13)",True,0.715439,1
11,dist_matrix[current][c] - instance['demands'][...,10,1.0,1100.8,0.033960,5.3,57,"(1100.8, 5.3, 1.0, 11)",True,0.722811,1
2,0.3 * dist_matrix[c][instance['depot']] + dist...,10,1.0,1112.0,0.030206,5.3,46,"(1112.0, 5.3, 1.0, 9)",True,0.737608,0


## Formal Benchmark & Ablation Runner

This section runs reproducible formal experiments (multi-seed + 5 ablation settings) and saves tables/plots automatically.

In [ ]:
import sys
from pathlib import Path

MODULE_DIR = Path("03_core_algorithm/modules").resolve()
if str(MODULE_DIR) not in sys.path:
    sys.path.append(str(MODULE_DIR))

from benchmark_experiment_workflow import (
    stratified_sample_instances,
    run_formal_experiments,
)

# Reproducible formal instance subset (small/medium/large balanced).
# Set FORMAL_PER_BUCKET to a larger number for a heavier benchmark.
FORMAL_PER_BUCKET = 10
FORMAL_SEED = 42
formal_instances = stratified_sample_instances(instances, per_bucket=FORMAL_PER_BUCKET, seed=FORMAL_SEED)

print(f"total loaded instances: {len(instances)}")
print(f"formal sampled instances: {len(formal_instances)}")
print(pd.Series([inst['num_nodes'] for inst in formal_instances]).describe())

ModuleNotFoundError: No module named 'benchmark_experiment_workflow'

In [ ]:
# P1 + P2 one-shot runner: baseline + formal ablation + export + plots.
# For stable reproducibility, you can start with generation_mode='mock'.
# If you want LLM mode, set env var CVRP_OPENAI_API_KEY and change generation_mode to 'llm'.

result_paths = run_formal_experiments(
    instances=formal_instances,
    seed_expressions=seed_expressions,
    evaluate_named_solver_on_instances=evaluate_named_solver_on_instances,
    summarize_expression_results=summarize_expression_results,
    nearest_neighbor_v2=nearest_neighbor_v2,
    greedy_cvrp_solver=greedy_cvrp_solver,
    ortools_cvrp_solver=ortools_cvrp_solver,
    evaluate_expression_list_on_instances=evaluate_expression_list_on_instances,
    dedup_expressions=dedup_expressions,
    filter_expressions_by_complexity=filter_expressions_by_complexity,
    expression_complexity=expression_complexity,
    add_novelty_columns=add_novelty_columns,
    sort_expression_summary=sort_expression_summary,
    update_archive_signatures=update_archive_signatures,
    generate_mock_candidates_from_top_expressions=generate_mock_candidates_from_top_expressions,
    generate_candidates_with_llm=generate_candidates_with_llm,
    output_root="04_experiment_outputs/formal_benchmark",
    run_prefix="base_cvrp",
    generation_mode="mock",   # switch to "llm" after setting CVRP_OPENAI_API_KEY
    llm_client=client,
    llm_model_name=DEFAULT_LLM_MODEL,
    llm_temperature=0.4,
    num_rounds=4,
    variants_per_expr=6,
    top_k_per_round=5,
    seeds=[42, 52, 62],
    verbose=True,
)

print(result_paths)

In [ ]:
ablation_seed_df = pd.read_csv(result_paths["ablation_seed_summary"])
ablation_agg_df = pd.read_csv(result_paths["ablation_aggregate_summary"])

display(ablation_seed_df.head())
display(ablation_agg_df)
print("plots folder:", result_paths["output_root"])